# Beam-search traces on Vamana graphs

Runs beam search over the graphs `gamma_sweep.py` built (stock Vamana and modified
Vamana at several gammas), records every node each query **expands**, and dumps the
traces to disk. Then it asks three questions of those traces:

1. **How often is each point expanded?** Is the work spread evenly, or do a few
   nodes get expanded by most queries?
2. **Who are the predecessors?** For each expanded node, which node put it on the
   queue, and is that usually the same node regardless of the query?
3. **Is the search query-independent?** How much of a query's trace is shared with
   every other query, and where in the trace does that shared part occur?

Sections 1-3 produce the trace files; sections 4-6 only read them, so they can be
rerun without searching again.

## Configuration

Graph paths follow `gamma_sweep.py`'s naming:
`<SWEEP_DIR>/graph-{vamana,modvamana}-<dataset>-<metric>-R<R>-alpha<alpha>[-gamma<g>]`.
Graphs that are not on disk are dropped with a warning.

In [ ]:
import heapq
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

# --- which graphs ----------------------------------------------------------
DATASET = "mnist-784-euclidean"     # hdf5 basename, as passed to gamma_sweep
SHORT   = "mnist"                   # gamma_sweep/<SHORT> output directory
METRIC  = "euclidean"
R       = 32
ALPHA   = 1.0
GAMMAS  = [0.9, 0.95, 0.99]         # modified-Vamana graphs to trace; [] = stock only

# --- paths -----------------------------------------------------------------
BASE_PATH  = Path("/scratch/pa2439/ANN-Search/datasets")
HDF5_PATH  = BASE_PATH / f"{DATASET}.hdf5"           # ground truth ('neighbors')
BASE_FBIN  = BASE_PATH / DATASET / "base.fbin"
QUERY_FBIN = BASE_PATH / DATASET / "query.fbin"
SWEEP_DIR  = Path("/scratch/pa2439/ANN-Search/navigable_graph_results/gamma_sweep") / SHORT
TRACE_DIR  = Path("/scratch/pa2439/ANN-Search/navigable_graph_results/search_traces") / SHORT

# --- search ----------------------------------------------------------------
BEAM_WIDTH = 64       # L: size of the beam (the best-L list)
K          = 10       # results returned per query, and recall@K
START      = 0        # entry point; ParlayANN's Vamana searches from node 0
N_QUERIES  = None     # None = every test query
RETRACE    = False    # True reruns search even when a trace file exists

# --- statistics ------------------------------------------------------------
TAUS           = [0.1, 0.25, 0.5, 0.9]  # "core" = nodes expanded by >= tau of queries
MIN_PRED_COUNT = 20       # predecessor shares only for nodes expanded this often
N_RANDOM_PAIRS = 3000     # query pairs for the trace-overlap (Jaccard) test
N_NN_PAIRS     = 1000     # ... plus this many (query, nearest other query) pairs
SEED           = 0

TAGBASE = f"{DATASET}-{METRIC}-R{R}-alpha{ALPHA:g}"
GRAPHS = {"vamana": SWEEP_DIR / f"graph-vamana-{TAGBASE}"}
GRAPHS.update({f"γ={g:g}": SWEEP_DIR / f"graph-modvamana-{TAGBASE}-gamma{g:g}"
               for g in GAMMAS})

for label, p in list(GRAPHS.items()):
    if not p.exists():
        print(f"missing, skipped: {label:8s} {p}")
        del GRAPHS[label]
if not GRAPHS:
    raise FileNotFoundError(f"none of the configured graphs exist under {SWEEP_DIR}")

TRACE_DIR.mkdir(parents=True, exist_ok=True)


def trace_path(graph_path):
    return TRACE_DIR / f"trace-{Path(graph_path).name}-L{BEAM_WIDTH}-k{K}.npz"


for label, p in GRAPHS.items():
    print(f"{label:8s} {p.name}\n{'':8s} -> {trace_path(p).name}")

## 1. Load vectors, queries and ground truth

The ann-benchmarks `neighbors` dataset is ground truth for the file's own metric. The
angular datasets (coco_i2i, glove25) are built and searched with euclidean distance
here, so their stored neighbors would be wrong. Ground truth is therefore read from the
hdf5 only when the dataset name ends in `METRIC`, and computed by brute force otherwise.

In [ ]:
def read_fbin(path):
    with open(path, "rb") as f:
        n, dims = np.fromfile(f, dtype=np.int32, count=2)
        return np.fromfile(f, dtype=np.float32).reshape(n, dims)


def read_csr(path):
    # ParlayANN graph file -> (indptr, neighbors)
    with open(path, "rb") as f:
        n, _max_deg = np.fromfile(f, dtype=np.uint32, count=2)
        sizes = np.fromfile(f, dtype=np.uint32, count=int(n)).astype(np.int64)
        edges = np.fromfile(f, dtype=np.uint32)
    indptr = np.zeros(len(sizes) + 1, dtype=np.int64)
    np.cumsum(sizes, out=indptr[1:])
    return indptr, edges.astype(np.int32)


def brute_force_gt(V, Q, k, chunk=256):
    # Exact k-NN under squared euclidean, nearest first.
    norms = np.einsum("ij,ij->i", V, V)
    out = np.empty((len(Q), k), dtype=np.int32)
    for s in tqdm(range(0, len(Q), chunk), desc="ground truth"):
        q = Q[s:s + chunk]
        D = norms[None, :] - 2.0 * (q @ V.T)          # + |q|^2, constant per row
        part = np.argpartition(D, k, axis=1)[:, :k]
        order = np.take_along_axis(D, part, axis=1).argsort(axis=1)
        out[s:s + chunk] = np.take_along_axis(part, order, axis=1)
    return out


V = read_fbin(BASE_FBIN)
Q = read_fbin(QUERY_FBIN)[:N_QUERIES]
N, NQ = len(V), len(Q)

if HDF5_PATH.exists() and DATASET.endswith(METRIC):
    import h5py
    with h5py.File(HDF5_PATH, "r") as f:
        GT = f["neighbors"][:NQ, :K].astype(np.int32)
    print(f"ground truth from {HDF5_PATH.name}")
else:
    GT = brute_force_gt(V, Q, K)

print(f"{N:,} points, {NQ:,} queries, dim {V.shape[1]}")

## 2. Beam search with a trace

The search keeps a min-heap of candidates and a max-heap holding the best `L` nodes
seen so far. Each step pops the closest candidate and stops if it is farther than the
worst of the best `L`. Otherwise the candidate is **expanded**: the distance is
computed for each out-neighbor that has not been seen yet, and those that beat the
current worst are pushed onto both heaps. This is the same rule as ParlayANN's beam
search and HNSW's `search_layer`, and it returns the same results.

Each expansion appends one trace row, `(node, predecessor, distance)`:

- **node**: the node expanded, in expansion order. A node is expanded at most once
  per query.
- **predecessor**: the node whose expansion pushed this one onto the queue. A node
  is scored once, the first time it is seen, so this is well defined. The start node
  has predecessor `-1`.
- **distance**: squared euclidean distance from the query.

The per-query count of distance computations (`n_dist`) is stored alongside, since it
measures work better than the number of expansions does.

In [ ]:
def traced_beam_search(q, V, indptr, nbrs, start, L, k, seen, done, stamp):
    # `seen` / `done` are length-n int arrays shared across queries; a node is
    # seen/expanded for this query when its entry equals `stamp`, so nothing
    # has to be cleared between queries.
    diff = V[start] - q
    d0 = float(diff @ diff)
    seen[start] = stamp
    cand = [(d0, start, -1)]      # min-heap of (dist, node, predecessor)
    best = [(-d0, start)]         # max-heap of the L closest seen so far
    nodes, preds, dists = [], [], []
    n_dist = 1

    while cand:
        du, u, p = heapq.heappop(cand)
        if len(best) >= L and du > -best[0][0]:
            break                                  # nothing left can improve the beam
        if done[u] == stamp:
            continue
        done[u] = stamp
        nodes.append(u); preds.append(p); dists.append(du)

        nb = nbrs[indptr[u]:indptr[u + 1]]
        nb = nb[seen[nb] != stamp]
        if len(nb) == 0:
            continue
        seen[nb] = stamp
        diff = V[nb] - q
        dn = np.einsum("ij,ij->i", diff, diff)
        n_dist += len(nb)

        worst = -best[0][0]
        for w, dw in zip(nb.tolist(), dn.tolist()):
            if len(best) < L or dw < worst:
                heapq.heappush(cand, (dw, w, u))
                heapq.heappush(best, (-dw, w))
                if len(best) > L:
                    heapq.heappop(best)
                worst = -best[0][0]

    top = [w for _, w in sorted((-nd, w) for nd, w in best)[:k]]
    return nodes, preds, dists, n_dist, top

In [ ]:
def trace_graph(graph_path, V, Q, GT, L, k, start):
    # Search every query and return the trace as flat CSR-style arrays.
    indptr, nbrs = read_csr(graph_path)
    if len(indptr) - 1 != len(V):
        raise ValueError(f"{graph_path.name} has {len(indptr) - 1} nodes, "
                         f"base.fbin has {len(V)}")

    seen = np.zeros(len(V), dtype=np.int32)
    done = np.zeros(len(V), dtype=np.int32)
    node, pred, dist = [], [], []
    q_ptr = np.zeros(len(Q) + 1, dtype=np.int64)
    n_dist = np.zeros(len(Q), dtype=np.int64)
    topk = np.full((len(Q), k), -1, dtype=np.int32)

    t0 = time.perf_counter()
    for i, q in enumerate(tqdm(Q, desc=graph_path.name[:40])):
        a, b, c, nd, top = traced_beam_search(q, V, indptr, nbrs, start, L, k,
                                              seen, done, i + 1)
        node.extend(a); pred.extend(b); dist.extend(c)
        q_ptr[i + 1] = len(node)
        n_dist[i] = nd
        topk[i, :len(top)] = top
    seconds = time.perf_counter() - t0

    kk = min(k, GT.shape[1])
    recall = (topk[:, :kk, None] == GT[:, None, :kk]).any(axis=2).sum(axis=1) / kk
    return {
        "q_ptr": q_ptr,
        "node": np.asarray(node, dtype=np.int32),
        "pred": np.asarray(pred, dtype=np.int32),
        "dist": np.asarray(dist, dtype=np.float32),
        "n_dist": n_dist,
        "topk": topk,
        "recall": recall.astype(np.float32),
        "meta": {"graph": str(graph_path), "base_fbin": str(BASE_FBIN),
                 "query_fbin": str(QUERY_FBIN), "n_points": len(V),
                 "n_queries": len(Q), "beam_width": L, "k": k, "start": start,
                 "metric": "squared euclidean", "search_seconds": seconds},
    }

## 3. Run and dump the traces

Each graph is written to one compressed `.npz` in `TRACE_DIR`. The layout is
CSR-style, so query `i`'s trace is rows `q_ptr[i]:q_ptr[i+1]`:

| array | shape | meaning |
|---|---|---|
| `q_ptr` | `(nq+1,)` int64 | row offsets per query |
| `node` | `(rows,)` int32 | node expanded, in expansion order |
| `pred` | `(rows,)` int32 | node that pushed it onto the queue (`-1` for the start) |
| `dist` | `(rows,)` float32 | squared distance to the query |
| `n_dist` | `(nq,)` int64 | distance computations for the query |
| `topk` | `(nq, K)` int32 | returned neighbors, nearest first |
| `recall` | `(nq,)` float32 | recall@K against ground truth |
| `meta` | JSON string | graph path, L, K, start, search time |

A trace that already exists is loaded instead of recomputed unless `RETRACE = True`.

In [ ]:
def save_trace(path, tr):
    arrays = {k: v for k, v in tr.items() if k != "meta"}
    np.savez_compressed(path, meta=np.array(json.dumps(tr["meta"])), **arrays)


def load_trace(path):
    with np.load(path) as f:
        tr = {k: f[k] for k in f.files if k != "meta"}
        tr["meta"] = json.loads(str(f["meta"]))
    return tr


traces = {}
for label, gpath in GRAPHS.items():
    out = trace_path(gpath)
    if out.exists() and not RETRACE:
        traces[label] = load_trace(out)
        print(f"{label:8s} loaded {out.name}")
        continue
    tr = trace_graph(gpath, V, Q, GT, BEAM_WIDTH, K, START)
    save_trace(out, tr)
    traces[label] = tr
    print(f"{label:8s} wrote  {out}  ({len(tr['node']):,} rows, "
          f"{tr['meta']['search_seconds']:.1f}s)")

LABELS = list(traces)
print()
print(pd.DataFrame({
    "graph": LABELS,
    f"recall@{K}": [traces[l]["recall"].mean() for l in LABELS],
    "mean expansions": [np.diff(traces[l]["q_ptr"]).mean() for l in LABELS],
    "mean dist comps": [traces[l]["n_dist"].mean() for l in LABELS],
}).round(3).to_string(index=False))

## 4. Per-graph quantities

These are computed once for each trace and reused by every plot below:

- `counts[v]`: the number of queries that expanded `v`. Each query expands a node at
  most once, so `counts / NQ` is the fraction of queries that expand `v`.
- `indeg[v]` and `depth[v]`: in-degree, and BFS hop distance from `START`. They give
  context for *which* nodes are popular.
- `pairs`: every `(predecessor, node)` edge the search used, with how many queries
  used it.

The plots that overlay graphs show up to four of them, each in a fixed color. Stock
Vamana is drawn as a dashed gray line, as in `Vamana-Runs.ipynb`.

In [ ]:
import matplotlib.pyplot as plt

C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e5e4e0"
plt.rcParams.update({
    "figure.dpi": 130, "font.size": 9,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
})

# Colour follows the graph, not its position in a particular plot.
PLOT = ([l for l in LABELS if l == "vamana"] + [l for l in LABELS if l != "vamana"])[:4]
_gamma_cols = iter(C)
STYLE = {l: (dict(color=MUTED, linestyle="--", linewidth=1.6) if l == "vamana"
             else dict(color=next(_gamma_cols), linewidth=2)) for l in PLOT}


def bfs_depth(indptr, nbrs, start):
    depth = np.full(len(indptr) - 1, -1, dtype=np.int32)
    depth[start] = 0
    frontier, d = np.array([start]), 0
    while len(frontier):
        d += 1
        nb = np.concatenate([nbrs[indptr[u]:indptr[u + 1]] for u in frontier])
        nb = np.unique(nb)
        nb = nb[depth[nb] < 0]
        depth[nb] = d
        frontier = nb
    return depth


def predecessor_pairs(tr, n):
    m = tr["pred"] >= 0
    key = tr["pred"][m].astype(np.int64) * n + tr["node"][m]
    keys, cnt = np.unique(key, return_counts=True)
    return pd.DataFrame({"pred": keys // n, "node": keys % n, "count": cnt})


def dominant_predecessor(pairs):
    # One row per node: its most frequent predecessor and that predecessor's share.
    g = pairs.groupby("node")["count"]
    top = (pairs.sort_values(["node", "count"], ascending=[True, False])
                .drop_duplicates("node").set_index("node"))
    return pd.DataFrame({"top_pred": top["pred"], "top_count": top["count"],
                         "total": g.sum(), "n_preds": g.size()}).assign(
        share=lambda d: d.top_count / d.total)


S = {}
for label in LABELS:
    tr = traces[label]
    indptr, nbrs = read_csr(GRAPHS[label])
    lengths = np.diff(tr["q_ptr"])
    pairs = predecessor_pairs(tr, N)
    S[label] = {
        "counts": np.bincount(tr["node"], minlength=N),
        "lengths": lengths,
        "qid": np.repeat(np.arange(NQ), lengths),
        "step": np.arange(len(tr["node"])) - np.repeat(tr["q_ptr"][:-1], lengths),
        "indeg": np.bincount(nbrs, minlength=N),
        "depth": bfs_depth(indptr, nbrs, START),
        "pairs": pairs,
        "dom": dominant_predecessor(pairs),
    }
    print(f"{label:8s} {len(pairs):,} distinct predecessor edges used, "
          f"{(S[label]['counts'] > 0).sum():,} distinct nodes expanded")

## 5. Statistics

### 5.1 How often is each point expanded?

**Rank-frequency.** Nodes are sorted by the fraction of queries that expand them. A
flat curve means work is spread evenly. A steep head followed by a long tail means a
small set of nodes is expanded by nearly every query.

**Concentration.** The share of all expansions that fall on the `m` most-expanded
nodes. The dotted line marks `m = 1%` of the points.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
for l in PLOT:
    c = np.sort(S[l]["counts"])[::-1]
    c = c[c > 0]
    rank = np.arange(1, len(c) + 1)
    axes[0].plot(rank, c / NQ, label=l, **STYLE[l])
    axes[1].plot(rank, np.cumsum(c) / c.sum(), label=l, **STYLE[l])

axes[0].set(xscale="log", yscale="log", xlabel="node rank",
            ylabel="fraction of queries expanding node")
axes[0].set_title("Expansion frequency by rank", fontsize=10)
axes[1].axvline(0.01 * N, color=MUTED, linestyle=":", linewidth=1)
axes[1].annotate("1% of points", xy=(0.01 * N, 0.02), xytext=(4, 0),
                 textcoords="offset points", fontsize=7.5, color=MUTED)
axes[1].set(xscale="log", ylim=(0, 1.02), xlabel="m most-expanded nodes",
            ylabel="share of all expansions")
axes[1].set_title("Share of expansions on the top-m nodes", fontsize=10)
axes[1].legend(frameon=False, fontsize=8, loc="upper left")
fig.tight_layout(); plt.show()

rows = []
for l in LABELS:
    c = np.sort(S[l]["counts"])[::-1]
    tot = c.sum()
    rows.append({
        "graph": l,
        "distinct expanded": f"{(c > 0).sum():,} ({(c > 0).mean():.1%})",
        "share on top 0.1%": c[:max(1, N // 1000)].sum() / tot,
        "share on top 1%": c[:max(1, N // 100)].sum() / tot,
        "share on top 10%": c[:max(1, N // 10)].sum() / tot,
        **{f"nodes in >= {t:g} of queries": int((c >= t * NQ).sum()) for t in TAUS},
    })
print(pd.DataFrame(rows).round(3).to_string(index=False))

**What the popular nodes look like.** The mean expansion frequency, binned by
in-degree and by BFS hop distance from the start. If popularity is mostly a question
of being close to `START`, the right panel falls steeply. If it is driven by hubs, the
left panel rises.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4))
for l in PLOT:
    s = S[l]
    f = s["counts"] / NQ
    df = pd.DataFrame({"indeg": s["indeg"], "depth": s["depth"], "f": f})
    by_deg = df.groupby("indeg")["f"].agg(["mean", "size"])
    by_deg = by_deg[by_deg["size"] >= 20]            # drop sparse, noisy bins
    axes[0].plot(by_deg.index, by_deg["mean"], label=l, **STYLE[l])
    by_dep = df[df.depth >= 0].groupby("depth")["f"].mean()
    by_dep = by_dep[by_dep > 0]                      # log axis: drop never-expanded depths
    axes[1].plot(by_dep.index, by_dep.values, marker="o", markersize=4, label=l, **STYLE[l])

axes[0].set(yscale="log", xlabel="in-degree", ylabel="mean fraction of queries")
axes[0].set_title("Expansion frequency vs in-degree", fontsize=10)
axes[1].set(yscale="log", xlabel=f"hops from start node {START}",
            ylabel="mean fraction of queries")
axes[1].set_title("Expansion frequency vs distance from start", fontsize=10)
axes[1].legend(frameon=False, fontsize=8)
fig.tight_layout(); plt.show()

focus = LABELS[0]
s = S[focus]
top = np.argsort(-s["counts"])[:15]
print(f"Most-expanded nodes, {focus}:")
print(pd.DataFrame({"node": top, "frac of queries": s["counts"][top] / NQ,
                    "in-degree": s["indeg"][top], "hops from start": s["depth"][top]}
                   ).round(3).to_string(index=False))

### 5.2 Most frequent predecessors

For each node `v`, the predecessor counts say through which node the search reached
`v`, summed over all queries that expanded it. The **dominant-predecessor share** is
the fraction of those queries that came through the single most common predecessor.
A share near 1 means `v` is entered through the same node no matter what the query
is. A low share means the route into `v` depends on the query.

The share is only computed for nodes expanded at least `MIN_PRED_COUNT` times. Below
that, a share of 1 mostly reflects a small sample.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4))
for l in PLOT:
    d = S[l]["dom"]
    d = d[d.total >= MIN_PRED_COUNT]
    if not len(d):
        continue
    x = np.sort(d.share.values)                      # ECDF: shares cluster at 1/2, 1/3, ...
    axes[0].step(x, np.arange(1, len(x) + 1) / len(x), where="post", label=l, **STYLE[l])
    # share against how often the node is expanded, in log-spaced count bins
    edges = np.unique(np.geomspace(MIN_PRED_COUNT, d.total.max() + 1, 12).astype(int))
    grp = pd.cut(d.total, edges, include_lowest=True)
    m = d.groupby(grp, observed=True).share.mean()
    mids = [iv.mid / NQ for iv in m.index]
    axes[1].plot(mids, m.values, marker="o", markersize=4, label=l, **STYLE[l])

axes[0].set(xlim=(0, 1.02), ylim=(0, 1.02), xlabel="dominant-predecessor share",
            ylabel="fraction of nodes with share <= x")
axes[0].set_title(f"Route concentration (nodes expanded >= {MIN_PRED_COUNT}x)",
                  fontsize=10)
axes[1].set(xscale="log", ylim=(0, 1.02), xlabel="fraction of queries expanding node",
            ylabel="mean dominant-predecessor share")
axes[1].set_title("Are popular nodes entered through one door?", fontsize=10)
axes[1].legend(frameon=False, fontsize=8)
fig.tight_layout(); plt.show()

In [ ]:
def top_predecessors(pairs, v, n=3):
    p = pairs[pairs.node == v].nlargest(n, "count")
    tot = pairs.loc[pairs.node == v, "count"].sum()
    return ", ".join(f"{int(r.pred)} ({r['count'] / tot:.0%})" for _, r in p.iterrows())


focus = LABELS[0]
s = S[focus]
pairs, dom = s["pairs"], s["dom"]

top = [v for v in np.argsort(-s["counts"]) if v != START][:15]
print(f"Most-expanded nodes and who added them, {focus}:")
print(pd.DataFrame({
    "node": top,
    "frac of queries": s["counts"][top] / NQ,
    "distinct preds": [int(dom.n_preds.get(v, 0)) for v in top],
    "top predecessors (share)": [top_predecessors(pairs, v) for v in top],
}).round(3).to_string(index=False))

# Nodes ranked by how many expansions they fed, i.e. how often they were the
# predecessor of an expanded node.
credit = np.bincount(traces[focus]["pred"][traces[focus]["pred"] >= 0], minlength=N)
fed = np.argsort(-credit)[:15]
print(f"\nNodes that most often added an expanded node to the queue, {focus}:")
print(pd.DataFrame({"node": fed, "expansions fed / query": credit[fed] / NQ,
                    "out-degree": np.diff(read_csr(GRAPHS[focus])[0])[fed],
                    "hops from start": s["depth"][fed]}).round(3).to_string(index=False))

print(f"\nMost-used edges (pred -> node), {focus}:")
e = pairs.nlargest(15, "count").assign(**{"frac of queries": lambda d: d["count"] / NQ})
print(e[["pred", "node", "frac of queries"]].round(3).to_string(index=False))

### 5.3 Is the search independent of the query?

The three views below get at this from different directions.

**Core share per query.** The *core* at threshold τ is the set of nodes expanded by at
least a τ fraction of queries. For each query, the core share is the fraction of its
own expansions that fall in the core. The table gives the per-query distribution at
the `TAUS` thresholds. The plot sweeps τ continuously and shows, next to it, how big
the core is at each τ. If a small high-τ core holds most of every query's trace, most
of the work is the same for every query.

**Commonness by step.** At step `t` of a trace, how many queries also expand the node
expanded there? The line is the median over queries and the band the interquartile
range, for the first graph. If the early steps are query-independent and the later
ones query-specific, the curve starts high and falls.

**Overlap vs query distance.** The Jaccard similarity of two queries' expanded sets,
over random query pairs plus each sampled query paired with its nearest other query,
binned by the distance between the two queries. If the trace did not depend on the
query, the line would be flat and high. A line that falls with distance means the
trace follows the query.

In [ ]:
rows = []
core_share = {}
for l in LABELS:
    s, tr = S[l], traces[l]
    for t in TAUS:
        core = s["counts"] >= t * NQ
        per_q = np.add.reduceat(core[tr["node"]].astype(np.float64), tr["q_ptr"][:-1]) / s["lengths"]
        core_share[(l, t)] = per_q
        rows.append({"graph": l, "tau": t, "core size": int(core.sum()),
                     "core % of points": core.mean() * 100,
                     "mean core share": per_q.mean(), "median core share": np.median(per_q),
                     "p10 core share": np.percentile(per_q, 10)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

# Core share over a continuous tau grid: mean share (left) and the size of the
# core that buys it (right).
tau_grid = np.geomspace(1.0 / NQ, 1.0, 40)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4))
for l in PLOT:
    s, tr = S[l], traces[l]
    frac = s["counts"][tr["node"]] / NQ               # commonness of each trace row
    mean_share = [(frac >= t).mean() for t in tau_grid]   # rows weighted by query length
    core_size = [(s["counts"] >= t * NQ).sum() for t in tau_grid]
    axes[0].plot(tau_grid, mean_share, label=l, **STYLE[l])
    axes[1].plot(tau_grid, core_size, label=l, **STYLE[l])
axes[0].set(xscale="log", ylim=(0, 1.02), xlabel="τ (core = nodes expanded by >= τ of queries)",
            ylabel="share of expansions in the core")
axes[0].set_title("How much of the search is shared work?", fontsize=10)
axes[1].set(xscale="log", yscale="log", xlabel="τ", ylabel="core size (nodes)")
axes[1].set_title("Size of the core", fontsize=10)
axes[1].legend(frameon=False, fontsize=8)
fig.tight_layout(); plt.show()

In [ ]:
def commonness_by_step(l):
    # Matrix (query, step) of counts[node]/NQ, padded with NaN past the trace end.
    s, tr = S[l], traces[l]
    T = int(np.percentile(s["lengths"], 95))
    M = np.full((NQ, T), np.nan)
    keep = s["step"] < T
    M[s["qid"][keep], s["step"][keep]] = s["counts"][tr["node"][keep]] / NQ
    alive = (~np.isnan(M)).sum(axis=0)
    ok = alive >= max(20, NQ // 20)                 # enough queries still running
    q25, q50, q75 = np.nanpercentile(M[:, ok], [25, 50, 75], axis=0)
    return np.arange(T)[ok], q25, q50, q75


fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4))
x, lo, mid, hi = commonness_by_step(LABELS[0])
axes[0].fill_between(x, lo, hi, color=C[0], alpha=0.18, linewidth=0)
axes[0].plot(x, mid, color=C[0], linewidth=2)
axes[0].set(ylim=(0, 1.02), xlabel="expansion step",
            ylabel="fraction of queries expanding this node")
axes[0].set_title(f"Commonness by step, {LABELS[0]} (median, IQR)", fontsize=10)
for l in PLOT:
    x, _, mid, _ = commonness_by_step(l)
    axes[1].plot(x, mid, label=l, **STYLE[l])
axes[1].set(ylim=(0, 1.02), xlabel="expansion step", ylabel="median fraction of queries")
axes[1].set_title("Commonness by step, all graphs", fontsize=10)
axes[1].legend(frameon=False, fontsize=8)
fig.tight_layout(); plt.show()

In [ ]:
def query_pairs(Q, n_random, n_nn, seed=SEED, chunk=512):
    # Random pairs, plus (query, its nearest other query) for a sample of queries.
    rng = np.random.default_rng(seed)
    I = rng.integers(0, len(Q), n_random)
    J = (I + rng.integers(1, len(Q), n_random)) % len(Q)   # J != I
    src = rng.choice(len(Q), min(n_nn, len(Q)), replace=False)
    qn = np.einsum("ij,ij->i", Q, Q)
    nn = np.empty(len(src), dtype=np.int64)
    for s in range(0, len(src), chunk):
        b = src[s:s + chunk]
        D = qn[None, :] - 2.0 * (Q[b] @ Q.T)
        D[np.arange(len(b)), b] = np.inf
        nn[s:s + chunk] = D.argmin(axis=1)
    I = np.concatenate([I, src]); J = np.concatenate([J, nn])
    kind = np.array(["random"] * n_random + ["nearest"] * len(src))
    d = np.sqrt(np.maximum(((Q[I] - Q[J]) ** 2).sum(axis=1), 0))
    return I, J, d, kind


def trace_jaccard(tr, I, J):
    sets = {}
    def get(i):
        if i not in sets:
            sets[i] = np.unique(tr["node"][tr["q_ptr"][i]:tr["q_ptr"][i + 1]])
        return sets[i]
    out = np.empty(len(I))
    for t, (i, j) in enumerate(zip(I, J)):
        a, b = get(i), get(j)
        inter = len(np.intersect1d(a, b, assume_unique=True))
        out[t] = inter / (len(a) + len(b) - inter)
    return out


PI, PJ, PD, PKIND = query_pairs(Q, N_RANDOM_PAIRS, N_NN_PAIRS)
JAC = {l: trace_jaccard(traces[l], PI, PJ) for l in LABELS}

edges = np.unique(np.quantile(PD, np.linspace(0, 1, 13)))
grp = np.clip(np.searchsorted(edges, PD, side="right") - 1, 0, len(edges) - 2)
fig, ax = plt.subplots(figsize=(6.2, 3.6))
for l in PLOT:
    df = pd.DataFrame({"g": grp, "d": PD, "j": JAC[l]}).groupby("g").agg(d=("d", "median"),
                                                                       j=("j", "mean"))
    ax.plot(df.d, df.j, marker="o", markersize=4, label=l, **STYLE[l])
ax.set(ylim=(0, 1.02), xlabel="distance between the two queries",
       ylabel="mean Jaccard of expanded sets")
ax.set_title("Trace overlap vs query distance", fontsize=10)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); plt.show()

print(pd.DataFrame({
    "graph": LABELS,
    "Jaccard, random pairs": [JAC[l][PKIND == "random"].mean() for l in LABELS],
    "Jaccard, nearest pairs": [JAC[l][PKIND == "nearest"].mean() for l in LABELS],
}).round(3).to_string(index=False))

## 6. Summary across graphs

One row per traced graph:

- `top-1% share`: the share of all expansions on the 1% most-expanded nodes.
- `core(0.5) share`: the mean, over queries, of the fraction of a query's expansions
  on nodes that at least half of all queries expand.
- `dom-pred share`: the median dominant-predecessor share over nodes expanded at least
  `MIN_PRED_COUNT` times.
- `Jaccard rand/nn`: the mean trace overlap for random query pairs and for
  nearest-neighbor query pairs.

In [ ]:
summary = []
for l in LABELS:
    s, tr = S[l], traces[l]
    c = np.sort(s["counts"])[::-1]
    d = s["dom"]; d = d[d.total >= MIN_PRED_COUNT]
    core = s["counts"] >= 0.5 * NQ
    summary.append({
        "graph": l,
        f"recall@{K}": tr["recall"].mean(),
        "expansions": s["lengths"].mean(),
        "dist comps": tr["n_dist"].mean(),
        "distinct expanded %": (c > 0).mean() * 100,
        "top-1% share": c[:max(1, N // 100)].sum() / c.sum(),
        "core(0.5) size": int(core.sum()),
        "core(0.5) share": core_share[(l, 0.5)].mean() if 0.5 in TAUS else np.nan,
        "dom-pred share": d.share.median() if len(d) else np.nan,
        "Jaccard rand": JAC[l][PKIND == "random"].mean(),
        "Jaccard nn": JAC[l][PKIND == "nearest"].mean(),
    })
summary = pd.DataFrame(summary)
summary.to_csv(TRACE_DIR / f"trace-summary-L{BEAM_WIDTH}-k{K}.csv", index=False)
print(summary.round(3).to_string(index=False))